# Phase 8 Tick Pipeline Visualization

This notebook uses the real `TickSimulationEngine` to compare how behavior, liquidity, volatility, price generation, activity, and quote generation affect complete market ticks. Each scenario uses the same seed schedule so differences come from the configured market condition rather than unrelated random samples.

In [ ]:
from datetime import datetime, timedelta
from html import escape
from pathlib import Path
import sys

from IPython.display import HTML, display

repo_root = Path.cwd()
if not (repo_root / 'app').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from app.simulation import (
    BehaviorType,
    DEFAULT_SIMULATED_STOCKS,
    MARKET_TIMEZONE,
    MarketBehaviorConfig,
    StockActivityConfig,
    TickSimulationConfig,
    TickSimulationEngine,
)

print(f'Python kernel: {sys.executable}')
print(f'Repository: {repo_root}')

## Controlled scenarios

The baseline uses normal behavior and medium liquidity. The other scenarios change exactly one main input: uptrend, downtrend, volatility spike, low liquidity, or high liquidity. We average 30 deterministic runs per scenario so directional behavior is visible above market noise.

In [ ]:
stock = DEFAULT_SIMULATED_STOCKS[0]  # AAPL
start_time = datetime(2026, 8, 31, 8, 30, tzinfo=MARKET_TIMEZONE)
steps = 120
step_size = timedelta(minutes=2)
runs = 30

scenarios = {
    'Baseline': {'liquidity': 0.5, 'behavior': BehaviorType.NORMAL},
    'Uptrend': {'liquidity': 0.5, 'behavior': BehaviorType.UPTREND},
    'Downtrend': {'liquidity': 0.5, 'behavior': BehaviorType.DOWNTREND},
    'Volatility spike': {'liquidity': 0.5, 'behavior': BehaviorType.VOLATILITY_SPIKE},
    'Low liquidity': {'liquidity': 0.1, 'behavior': BehaviorType.NORMAL},
    'High liquidity': {'liquidity': 0.9, 'behavior': BehaviorType.NORMAL},
}

def run_scenario(name, settings):
    totals = {key: [0.0] * steps for key in ('price_index', 'spread_bps', 'volume', 'depth')}
    sample_ticks = None
    for run in range(runs):
        engine = TickSimulationEngine(TickSimulationConfig(seed=1000 + run))
        engine.configure_stock(stock.symbol, StockActivityConfig(liquidity=settings['liquidity']))
        engine.add_behavior_from_config(
            MarketBehaviorConfig(
                symbol=stock.symbol,
                behavior_type=settings['behavior'],
                duration=steps * step_size + timedelta(minutes=1),
                strength=1.0,
            ),
            current_time=start_time,
        )
        ticks = engine.simulate([stock], start_time=start_time, steps=steps, step_size=step_size)
        sample_ticks = sample_ticks or ticks
        for index, tick in enumerate(ticks):
            price = float(tick.price)
            totals['price_index'][index] += price / float(stock.starting_price) * 100
            totals['spread_bps'][index] += float(tick.ask - tick.bid) / price * 10_000
            totals['volume'][index] += tick.trade_volume
            totals['depth'][index] += tick.bid_size + tick.ask_size
    return {
        'name': name,
        'minutes': [(index + 1) * step_size.total_seconds() / 60 for index in range(steps)],
        **{key: [value / runs for value in values] for key, values in totals.items()},
        'sample_ticks': sample_ticks,
    }

results = [run_scenario(name, settings) for name, settings in scenarios.items()]
print(f'Generated {len(results)} scenarios × {runs} runs × {steps} ticks = {len(results) * runs * steps:,} ticks')

## Comparison charts

Price shows the behavior and price-engine result. Spread shows quote-engine response to liquidity and volatility. Volume shows activity-engine output. Top-of-book depth is bid size plus ask size and shows the liquidity effect on quote sizes.

In [ ]:
colors = {
    'Baseline': '#334155', 'Uptrend': '#16a34a', 'Downtrend': '#dc2626',
    'Volatility spike': '#9333ea', 'Low liquidity': '#ea580c', 'High liquidity': '#0284c7',
}

def render_chart(metric, title, y_label, width=980, height=430):
    margin = {'top': 58, 'right': 155, 'bottom': 55, 'left': 78}
    inner_width = width - margin['left'] - margin['right']
    inner_height = height - margin['top'] - margin['bottom']
    all_y = [value for result in results for value in result[metric]]
    min_y, max_y = min(all_y), max(all_y)
    padding = max((max_y - min_y) * 0.1, abs(max_y) * 0.002, 0.001)
    min_y, max_y = min_y - padding, max_y + padding
    max_x = max(results[0]['minutes'])

    def sx(value): return margin['left'] + value / max_x * inner_width
    def sy(value): return margin['top'] + (1 - (value - min_y) / (max_y - min_y)) * inner_height

    parts = [
        f'<svg viewBox="0 0 {width} {height}" width="100%" style="font-family:system-ui,sans-serif;max-width:100%">',
        f'<rect x="{margin["left"]}" y="{margin["top"]}" width="{inner_width}" height="{inner_height}" fill="white" stroke="#cbd5e1"/>',
        f'<text x="20" y="26" font-size="18" font-weight="650" fill="#0f172a">{escape(title)}</text>',
        f'<text x="20" y="45" font-size="11" fill="#64748b">AAPL · averages across {runs} same-seed comparisons</text>',
    ]
    for index in range(6):
        y_value = min_y + (max_y - min_y) * index / 5
        y = sy(y_value)
        parts += [f'<line x1="{margin["left"]}" y1="{y:.2f}" x2="{margin["left"] + inner_width}" y2="{y:.2f}" stroke="#e2e8f0"/>', f'<text x="{margin["left"] - 8}" y="{y + 4:.2f}" text-anchor="end" font-size="10" fill="#475569">{y_value:.2f}</text>']
    for minute in (0, 60, 120, 180, 240):
        x = sx(minute)
        parts += [f'<line x1="{x:.2f}" y1="{margin["top"]}" x2="{x:.2f}" y2="{margin["top"] + inner_height}" stroke="#f1f5f9"/>', f'<text x="{x:.2f}" y="{height - 25}" text-anchor="middle" font-size="10" fill="#475569">{minute}</text>']
    for row, result in enumerate(results):
        color = colors[result['name']]
        points = ' '.join(f'{sx(x):.2f},{sy(y):.2f}' for x, y in zip(result['minutes'], result[metric]))
        parts.append(f'<polyline points="{points}" fill="none" stroke="{color}" stroke-width="2"/>')
        legend_y = margin['top'] + 18 + row * 24
        parts += [f'<line x1="{margin["left"] + inner_width + 12}" y1="{legend_y}" x2="{margin["left"] + inner_width + 34}" y2="{legend_y}" stroke="{color}" stroke-width="3"/>', f'<text x="{margin["left"] + inner_width + 40}" y="{legend_y + 4}" font-size="11" fill="#334155">{escape(result["name"])}</text>']
    parts += [f'<text x="{margin["left"] + inner_width / 2}" y="{height - 7}" text-anchor="middle" font-size="11">Simulated minutes</text>', f'<text transform="translate(16 {margin["top"] + inner_height / 2}) rotate(-90)" text-anchor="middle" font-size="11">{escape(y_label)}</text>', '</svg>']
    return HTML(''.join(parts))

In [ ]:
display(render_chart('price_index', 'Behavior + liquidity → price engine', 'Indexed price (start = 100)'))
display(render_chart('spread_bps', 'Volatility + liquidity + price → quote engine', 'Spread (basis points)'))
display(render_chart('volume', 'Liquidity → activity engine', 'Average shares per tick'))
display(render_chart('depth', 'Liquidity → bid and ask sizes', 'Average top-of-book shares'))

## Scenario summary

The table condenses the same output. Final indexed price emphasizes directional behavior; average absolute movement emphasizes volatility; spread, volume, and depth expose quote and activity effects.

In [ ]:
rows = []
for result in results:
    moves = [abs(result['price_index'][i] - result['price_index'][i - 1]) for i in range(1, steps)]
    rows.append({
        'Scenario': result['name'],
        'Final price index': result['price_index'][-1],
        'Avg abs price move': sum(moves) / len(moves),
        'Avg spread (bps)': sum(result['spread_bps']) / steps,
        'Avg trade volume': sum(result['volume']) / steps,
        'Avg book depth': sum(result['depth']) / steps,
    })
headers = list(rows[0])
table = ['<table style="border-collapse:collapse;font-family:system-ui;font-size:13px"><thead><tr>']
table += [f'<th style="padding:7px 10px;border-bottom:2px solid #94a3b8;text-align:right">{escape(header)}</th>' for header in headers]
table.append('</tr></thead><tbody>')
for row in rows:
    table.append('<tr>')
    for header in headers:
        value = row[header]
        shown = value if isinstance(value, str) else f'{value:,.3f}'
        table.append(f'<td style="padding:7px 10px;border-bottom:1px solid #e2e8f0;text-align:right">{escape(str(shown))}</td>')
    table.append('</tr>')
table.append('</tbody></table>')
display(HTML(''.join(table)))

## Inspect complete ticks

These are actual validated `MarketTick` records from the baseline sample run. They show how price, bid, ask, sizes, volume, timestamp, and sequence number arrive together.

In [ ]:
[tick.model_dump(mode='json') for tick in results[0]['sample_ticks'][:5]]

## Reading the results

- **Uptrend and downtrend:** change drift before the price engine runs, so their averaged price paths separate from the baseline.
- **Volatility spike:** increases behavior-adjusted volatility, producing larger price fluctuations and wider quotes.
- **Low liquidity:** produces less volume, smaller quote sizes, wider spreads, and a larger price-volatility multiplier.
- **High liquidity:** produces more volume, larger quote sizes, narrower spreads, and smoother pricing.
- **Determinism:** rerunning the notebook with unchanged inputs produces the same charts and table.